**04 | Subject-Independent Data Splitting**

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Apply existing HARTH loading, windowing and feature extraction to **eight different participants**, then create reproducible, strictly participant-disjoint Train/Validation/Test partitions. This is an educational multi-person subset, **not** the full benchmark. No model fitting is performed.

**01. Imports and reusable Python functions**

Code comments identify the role of each operation. The algorithms remain in dedicated `.py` modules.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Find the repository and import the reusable functions.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root: raise RuntimeError("Not inside the project")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))
from core.signal_preprocessing import create_signal_windows
from core.feature_engineering import extract_signal_features
from core.data_splitting import split_group_indices
from harth_data import ACTIVITY_NAMES, ensure_harth_sample, load_harth_subject
print("Shared Python functions: ready")

Shared Python functions: ready


**02. Acquire a bounded, multi-participant HARTH subset**

Only selected v2.0 files are downloaded. Together the eight files require approximately 124 MiB. Files are kept under ignored `data/raw/`; large datasets are not committed.

In [2]:
# Select source-verified independent subjects.
SUBJECTS = ["S023", "S024", "S025", "S027", "S028", "S032", "S034", "S035"]
SOURCE = root / "data/raw/harth_v2"
paths = {s: ensure_harth_sample(SOURCE, subject=s) for s in SUBJECTS}
display(pd.DataFrame([
    {"subject": s, "MiB": round(paths[s].stat().st_size/1048576,2)}
    for s in SUBJECTS
]))

Using cached HARTH data: D:\human-movement-intelligence\data\raw\harth_v2\S023.csv
Downloaded 16,722,098 bytes to D:\human-movement-intelligence\data\raw\harth_v2\S024.csv
Downloaded 21,695,989 bytes to D:\human-movement-intelligence\data\raw\harth_v2\S025.csv
Downloaded 15,873,565 bytes to D:\human-movement-intelligence\data\raw\harth_v2\S027.csv
Downloaded 16,949,909 bytes to D:\human-movement-intelligence\data\raw\harth_v2\S028.csv
Downloaded 14,821,095 bytes to D:\human-movement-intelligence\data\raw\harth_v2\S032.csv
Downloaded 15,031,875 bytes to D:\human-movement-intelligence\data\raw\harth_v2\S034.csv
Downloaded 14,899,779 bytes to D:\human-movement-intelligence\data\raw\harth_v2\S035.csv


,subject,MiB
0,S023,11.50
1,S024,15.95
2,S025,20.69
3,S027,15.14
4,S028,16.16
5,S032,14.13
6,S034,14.34
7,S035,14.21


**03. Window and extract features separately for each person**

The same 2-second windows and five feature methods from Notebooks 02 and 03 are used. This guarantees windows never cross participant boundaries.

**Label provenance note:** In the real `S032.csv`, codes `10` and `11` appear but are not defined in the authors' HARTH v2.0 activity table. For this inspection-only notebook, they remain unchanged and are explicitly flagged. Neither code is assigned an invented activity meaning.

In [3]:
feature_blocks, metadata_blocks, subjects_summary = [], [], []
for subject in SUBJECTS:
    # Process each participant independently before combining rows.
    # Retain source-unknown activity codes only for the metadata audit.
    data, channels = load_harth_subject(
        paths[subject], allow_unmapped_labels=True
    )
    unknown_mask = ~data["label"].isin(ACTIVITY_NAMES)
    unknown_codes = sorted(data.loc[unknown_mask, "label"].unique().tolist())
    if unknown_codes:
        print(f"{subject}: undocumented HARTH codes = {unknown_codes}")
    X, y, info = create_signal_windows(
        data[channels].to_numpy(), data["label"].to_numpy(),
        window_size=100, step_size=50,
        timestamps=data["timestamp"], sampling_rate=50,
        min_label_purity=1.0, subject_id=subject,
    )
    if not len(X): raise ValueError(f"No valid windows for {subject}")
    features = extract_signal_features(X, channel_names=channels)
    feature_blocks.append(features)
    metadata_blocks.append(info)
    subjects_summary.append({"subject":subject,"raw_samples":len(data),
                             "valid_windows":len(y),"classes":len(np.unique(y)),
                             "unmapped_rows":int(unknown_mask.sum()),
                             "unmapped_codes":str(unknown_codes)})
# Preserve identical row ordering in feature and metadata tables.
feature_table = pd.concat(feature_blocks, ignore_index=True)
window_metadata = pd.concat(metadata_blocks, ignore_index=True)
assert len(feature_table) == len(window_metadata)
display(pd.DataFrame(subjects_summary))
print("Combined feature matrix:",feature_table.shape)

HARTH subject S023: 137,646 rows, 6 sensors
Input signal: 137,646 samples, 6 channels
Candidate windows: 2,751
Retained windows: 2,567 | Shape: (2567, 6, 100)
Rejected candidates: 184
Input windows: (2567, 6, 100)
Feature matrix: (2567, 30) | Methods: mean, std, rms, peak_to_peak, waveform_length
HARTH subject S024: 170,534 rows, 6 sensors
Input signal: 170,534 samples, 6 channels
Candidate windows: 3,409
Retained windows: 3,119 | Shape: (3119, 6, 100)
Rejected candidates: 290
Input windows: (3119, 6, 100)
Feature matrix: (3119, 30) | Methods: mean, std, rms, peak_to_peak, waveform_length
HARTH subject S025: 231,729 rows, 6 sensors
Input signal: 231,729 samples, 6 channels
Candidate windows: 4,633
Retained windows: 4,272 | Shape: (4272, 6, 100)
Rejected candidates: 361
Input windows: (4272, 6, 100)
Feature matrix: (4272, 30) | Methods: mean, std, rms, peak_to_peak, waveform_length
HARTH subject S027: 158,584 rows, 6 sensors
Input signal: 158,584 samples, 6 channels
Candidate windows: 3

ValueError: Unexpected HARTH activity codes: [10, 11]

## 04. Split participant IDs, not overlapping windows

Target allocation is 4 Train, 2 Validation and 2 Test individuals. Fractions apply to numbers of participants, not numbers of windows.

In [ ]:
# Use a deterministic random seed and allocate whole subjects.
partitions = split_group_indices(
    window_metadata["subject_id"].to_numpy(),
    train_fraction=0.50, validation_fraction=0.25,
    test_fraction=0.25, random_state=42,
)
groups = {
    name: set(window_metadata.iloc[index]["subject_id"])
    for name,index in partitions.items()
}
# Verify complete partition coverage and absence of identity leakage.
assert not (groups["train"]&groups["validation"] or
            groups["train"]&groups["test"] or groups["validation"]&groups["test"])
np.testing.assert_array_equal(
    np.sort(np.concatenate(list(partitions.values()))),
    np.arange(len(window_metadata))
)
display(pd.DataFrame([
    {"partition":p,"subjects":len(groups[p]),"windows":len(ix),
     "IDs":", ".join(sorted(groups[p]))}
    for p,ix in partitions.items()
]))
print("Subject leakage: NONE")

## 05. Audit class distributions across independent partitions

Subject-based splitting does not guarantee every activity class occurs in every partition. Display coverage before designing a valid classification experiment.

In [ ]:
# Count windows by observed activity and partition.
counts = pd.DataFrame({
    part: window_metadata.iloc[ix]["label"].value_counts()
    for part,ix in partitions.items()
}).fillna(0).astype(int).sort_index()
counts.index = [f"{int(k)}: {ACTIVITY_NAMES.get(int(k),'UNDOCUMENTED source code')}"
                for k in counts.index]
display(counts)
fractions = counts.div(counts.sum(axis=0),axis=1)*100
ax = fractions.T.plot.bar(stacked=True,figsize=(10,5))
ax.set(title="HARTH: grouped-split activity distribution",
       xlabel="Partition",ylabel="Fraction of windows (%)")
ax.legend(bbox_to_anchor=(1.02,1),loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("Classes absent by partition:",
      {p:counts.index[counts[p]==0].tolist() for p in counts.columns})
# Report source codes absent from the documented HARTH activity dictionary.
unmapped_window_codes = sorted(
    set(window_metadata["label"].unique()) - set(ACTIVITY_NAMES)
)
print("Undocumented codes in retained windows:", unmapped_window_codes)
print("Classification with undocumented codes: NOT APPROVED")


## 06. Final quality gate

No model fitting or learned preprocessing has occurred. Train-only scaling and a classification baseline require careful review of class coverage first.

In [ ]:
# Check deterministic features, complete grouping and valid shapes.
assert feature_table.shape[1] == 30
assert np.isfinite(feature_table.to_numpy()).all()
assert window_metadata["subject_id"].nunique() == 8
assert len(feature_table) == sum(len(ix) for ix in partitions.values())
for part,ix in partitions.items():
    print(f"{part}: X={feature_table.iloc[ix].shape}, y={window_metadata.iloc[ix]['label'].shape}")
print("Subject-independent splitting: PASSED")
print("Feature scaling trained: NO | Classifier trained: NO")
# The group split can be checked, but annotation provenance remains unresolved.
print("Undocumented label review required:", bool(unmapped_window_codes))
